# Loja do Seu Zé
Banco de dados PostgreSQL para gerenciar clientes, produtos, categorias, fornecedores, vendas, pagamentos e controle de estoque.

## Instalação e configuração do PostgreSQL no Colab

In [ ]:
# Instalação do PostgreSQL
!sudo apt update -qq
!sudo apt install -y -qq postgresql postgresql-contrib

10 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
postgresql is already the newest version (16+257build1.1).
postgresql-contrib is already the newest version (16+257build1.1).
0 upgraded, 0 newly installed, 0 to remove and 10 not upgraded.


In [ ]:
!pip install -q psycopg2-binary

In [ ]:
# Iniciar o servidor
!sudo service postgresql start

[ OK ]


In [ ]:
!sudo service postgresql status

16/main (port 5432): online


## Criação do banco de dados

In [ ]:
!sudo -u postgres createdb lojadoseuze

createdb: error: database creation failed: ERROR:  database "lojadoseuze" already exists


In [ ]:
# Atalho para rodar comandos SQL no banco lojadoseuze
!echo '#!/bin/bash' > /usr/local/bin/posql
!echo 'sudo -u postgres psql "$@" -d lojadoseuze & sudo -u postgres psql -c "\\q"' >> /usr/local/bin/posql
!chmod +x /usr/local/bin/posql

In [ ]:
!posql -c "select current_database();"

 current_database 
------------------
 lojadoseuze
(1 row)



## Criando as tabelas

Ordem de criação respeita as dependências de chave estrangeira:

`categoria` → `cliente` → `fornecedor` → `produto` → `venda` → `item_venda` → `compra` → `pagamento_venda`

In [ ]:
%%bash
posql -c 'CREATE TABLE categoria(
    codigo_categoria INTEGER PRIMARY KEY,
    nome VARCHAR NOT NULL
)'

ERROR:  relation "categoria" already exists


In [ ]:
%%bash
posql -c 'CREATE TABLE cliente(
    codigo_cliente INTEGER PRIMARY KEY,
    nome VARCHAR NOT NULL,
    cpf_cnpj VARCHAR UNIQUE NOT NULL,
    email VARCHAR UNIQUE NOT NULL
)'

ERROR:  relation "cliente" already exists


In [ ]:
%%bash
posql -c 'CREATE TABLE fornecedor(
    codigo_fornecedor INTEGER PRIMARY KEY,
    razao_social VARCHAR NOT NULL,
    cnpj VARCHAR UNIQUE NOT NULL,
    telefone VARCHAR
)'

ERROR:  relation "fornecedor" already exists


In [ ]:
%%bash
posql -c 'CREATE TABLE produto(
    codigo_produto INTEGER PRIMARY KEY,
    nome VARCHAR NOT NULL,
    preco_venda_atual DECIMAL NOT NULL,
    quantidade_estoque INTEGER NOT NULL,
    codigo_categoria INTEGER NOT NULL,
    CONSTRAINT fk_produto_categoria FOREIGN KEY (codigo_categoria)
        REFERENCES categoria(codigo_categoria)
)'

ERROR:  relation "produto" already exists


In [ ]:
%%bash
posql -c 'CREATE TABLE venda(
    codigo_venda INTEGER PRIMARY KEY,
    data_hora TIMESTAMP,
    valor_total DECIMAL,
    status_venda VARCHAR NOT NULL DEFAULT '\''PENDENTE'\'',
    codigo_cliente INTEGER,
    CONSTRAINT fk_venda_cliente FOREIGN KEY (codigo_cliente)
        REFERENCES cliente(codigo_cliente)
)'

ERROR:  relation "venda" already exists


In [ ]:
%%bash
posql -c 'CREATE TABLE item_venda(
    codigo_venda INTEGER NOT NULL,
    codigo_produto INTEGER NOT NULL,
    quantidade INTEGER NOT NULL,
    preco_unitario_gravado DECIMAL NOT NULL,
    PRIMARY KEY (codigo_venda, codigo_produto),
    CONSTRAINT fk_item_venda_venda FOREIGN KEY (codigo_venda)
        REFERENCES venda(codigo_venda),
    CONSTRAINT fk_item_venda_produto FOREIGN KEY (codigo_produto)
        REFERENCES produto(codigo_produto)
)'

ERROR:  relation "item_venda" already exists


In [ ]:
%%bash
posql -c 'CREATE TABLE compra(
    codigo_compra INTEGER PRIMARY KEY,
    codigo_fornecedor INTEGER NOT NULL,
    codigo_produto INTEGER NOT NULL,
    quantidade_comprada INTEGER NOT NULL,
    preco_custo_unitario DECIMAL NOT NULL,
    data_compra DATE NOT NULL,
    CONSTRAINT fk_compra_fornecedor FOREIGN KEY (codigo_fornecedor)
        REFERENCES fornecedor(codigo_fornecedor),
    CONSTRAINT fk_compra_produto FOREIGN KEY (codigo_produto)
        REFERENCES produto(codigo_produto)
)'

ERROR:  relation "compra" already exists


In [ ]:
%%bash
posql -c 'CREATE TABLE pagamento_venda(
    codigo_pagamento INTEGER PRIMARY KEY,
    codigo_venda INTEGER NOT NULL,
    forma_pagamento VARCHAR NOT NULL,
    valor_pago DECIMAL NOT NULL,
    CONSTRAINT fk_pagamento_venda FOREIGN KEY (codigo_venda)
        REFERENCES venda(codigo_venda)
)'

ERROR:  relation "pagamento_venda" already exists


### Verificando as tabelas criadas

In [ ]:
!posql -c "\\d"

              List of relations
 Schema |      Name       | Type  |  Owner   
--------+-----------------+-------+----------
 public | categoria       | table | postgres
 public | cliente         | table | postgres
 public | compra          | table | postgres
 public | fornecedor      | table | postgres
 public | item_venda      | table | postgres
 public | pagamento_venda | table | postgres
 public | produto         | table | postgres
 public | venda           | table | postgres
(8 rows)



In [ ]:
!posql -c "\\d categoria"

                       Table "public.categoria"
      Column      |       Type        | Collation | Nullable | Default 
------------------+-------------------+-----------+----------+---------
 codigo_categoria | integer           |           | not null | 
 nome             | character varying |           | not null | 
Indexes:
    "categoria_pkey" PRIMARY KEY, btree (codigo_categoria)
Referenced by:
    TABLE "produto" CONSTRAINT "fk_produto_categoria" FOREIGN KEY (codigo_categoria) REFERENCES categoria(codigo_categoria)



In [ ]:
!posql -c "\\d cliente"

                       Table "public.cliente"
     Column     |       Type        | Collation | Nullable | Default 
----------------+-------------------+-----------+----------+---------
 codigo_cliente | integer           |           | not null | 
 nome           | character varying |           | not null | 
 cpf_cnpj       | character varying |           | not null | 
 email          | character varying |           | not null | 
Indexes:
    "cliente_pkey" PRIMARY KEY, btree (codigo_cliente)
    "cliente_cpf_cnpj_key" UNIQUE CONSTRAINT, btree (cpf_cnpj)
    "cliente_email_key" UNIQUE CONSTRAINT, btree (email)
Referenced by:
    TABLE "venda" CONSTRAINT "fk_venda_cliente" FOREIGN KEY (codigo_cliente) REFERENCES cliente(codigo_cliente)



In [ ]:
!posql -c "\\d fornecedor"

                       Table "public.fornecedor"
      Column       |       Type        | Collation | Nullable | Default 
-------------------+-------------------+-----------+----------+---------
 codigo_fornecedor | integer           |           | not null | 
 razao_social      | character varying |           | not null | 
 cnpj              | character varying |           | not null | 
 telefone          | character varying |           |          | 
Indexes:
    "fornecedor_pkey" PRIMARY KEY, btree (codigo_fornecedor)
    "fornecedor_cnpj_key" UNIQUE CONSTRAINT, btree (cnpj)
Referenced by:
    TABLE "compra" CONSTRAINT "fk_compra_fornecedor" FOREIGN KEY (codigo_fornecedor) REFERENCES fornecedor(codigo_fornecedor)



In [ ]:
!posql -c "\\d produto"

                         Table "public.produto"
       Column       |       Type        | Collation | Nullable | Default 
--------------------+-------------------+-----------+----------+---------
 codigo_produto     | integer           |           | not null | 
 nome               | character varying |           | not null | 
 preco_venda_atual  | numeric           |           | not null | 
 quantidade_estoque | integer           |           | not null | 
 codigo_categoria   | integer           |           | not null | 
Indexes:
    "produto_pkey" PRIMARY KEY, btree (codigo_produto)
Foreign-key constraints:
    "fk_produto_categoria" FOREIGN KEY (codigo_categoria) REFERENCES categoria(codigo_categoria)
Referenced by:
    TABLE "compra" CONSTRAINT "fk_compra_produto" FOREIGN KEY (codigo_produto) REFERENCES produto(codigo_produto)
    TABLE "item_venda" CONSTRAINT "fk_item_venda_produto" FOREIGN KEY (codigo_produto) REFERENCES produto(codigo_produto)



In [ ]:
!posql -c "\\d venda"

                                        Table "public.venda"
     Column     |            Type             | Collation | Nullable |            Default            
----------------+-----------------------------+-----------+----------+-------------------------------
 codigo_venda   | integer                     |           | not null | 
 data_hora      | timestamp without time zone |           |          | 
 valor_total    | numeric                     |           |          | 
 status_venda   | character varying           |           | not null | 'PENDENTE'::character varying
 codigo_cliente | integer                     |           |          | 
Indexes:
    "venda_pkey" PRIMARY KEY, btree (codigo_venda)
Foreign-key constraints:
    "fk_venda_cliente" FOREIGN KEY (codigo_cliente) REFERENCES cliente(codigo_cliente)
Referenced by:
    TABLE "item_venda" CONSTRAINT "fk_item_venda_venda" FOREIGN KEY (codigo_venda) REFERENCES venda(codigo_venda)
    TABLE "pagamento_venda" CONSTRAINT "fk_pa

In [ ]:
!posql -c "\\d item_venda"

                     Table "public.item_venda"
         Column         |  Type   | Collation | Nullable | Default 
------------------------+---------+-----------+----------+---------
 codigo_venda           | integer |           | not null | 
 codigo_produto         | integer |           | not null | 
 quantidade             | integer |           | not null | 
 preco_unitario_gravado | numeric |           | not null | 
Indexes:
    "item_venda_pkey" PRIMARY KEY, btree (codigo_venda, codigo_produto)
Foreign-key constraints:
    "fk_item_venda_produto" FOREIGN KEY (codigo_produto) REFERENCES produto(codigo_produto)
    "fk_item_venda_venda" FOREIGN KEY (codigo_venda) REFERENCES venda(codigo_venda)



In [ ]:
!posql -c "\\d compra"

                      Table "public.compra"
        Column        |  Type   | Collation | Nullable | Default 
----------------------+---------+-----------+----------+---------
 codigo_compra        | integer |           | not null | 
 codigo_fornecedor    | integer |           | not null | 
 codigo_produto       | integer |           | not null | 
 quantidade_comprada  | integer |           | not null | 
 preco_custo_unitario | numeric |           | not null | 
 data_compra          | date    |           | not null | 
Indexes:
    "compra_pkey" PRIMARY KEY, btree (codigo_compra)
Foreign-key constraints:
    "fk_compra_fornecedor" FOREIGN KEY (codigo_fornecedor) REFERENCES fornecedor(codigo_fornecedor)
    "fk_compra_produto" FOREIGN KEY (codigo_produto) REFERENCES produto(codigo_produto)



In [ ]:
!posql -c "\\d pagamento_venda"

                    Table "public.pagamento_venda"
      Column      |       Type        | Collation | Nullable | Default 
------------------+-------------------+-----------+----------+---------
 codigo_pagamento | integer           |           | not null | 
 codigo_venda     | integer           |           | not null | 
 forma_pagamento  | character varying |           | not null | 
 valor_pago       | numeric           |           | not null | 
Indexes:
    "pagamento_venda_pkey" PRIMARY KEY, btree (codigo_pagamento)
Foreign-key constraints:
    "fk_pagamento_venda" FOREIGN KEY (codigo_venda) REFERENCES venda(codigo_venda)



## Inserindo dados

A ordem de inserção segue a mesma lógica: tabelas referenciadas primeiro.

### Categorias

In [ ]:
%%bash
posql -c "INSERT INTO categoria VALUES (1, 'Bebida');"
posql -c "INSERT INTO categoria VALUES (2, 'Fruta');"
posql -c "INSERT INTO categoria VALUES (3, 'Cigarro');"
posql -c "INSERT INTO categoria VALUES (4, 'Salgado');"
posql -c "INSERT INTO categoria VALUES (5, 'Doce');"
posql -c "INSERT INTO categoria VALUES (6, 'Laticínio');"


ERROR:  duplicate key value violates unique constraint "categoria_pkey"
DETAIL:  Key (codigo_categoria)=(1) already exists.
ERROR:  duplicate key value violates unique constraint "categoria_pkey"
DETAIL:  Key (codigo_categoria)=(2) already exists.
ERROR:  duplicate key value violates unique constraint "categoria_pkey"
DETAIL:  Key (codigo_categoria)=(3) already exists.
ERROR:  duplicate key value violates unique constraint "categoria_pkey"
DETAIL:  Key (codigo_categoria)=(4) already exists.
ERROR:  duplicate key value violates unique constraint "categoria_pkey"
DETAIL:  Key (codigo_categoria)=(5) already exists.
ERROR:  duplicate key value violates unique constraint "categoria_pkey"
DETAIL:  Key (codigo_categoria)=(6) already exists.


In [ ]:
!posql -c "SELECT * FROM categoria;"

 codigo_categoria |   nome    
------------------+-----------
                1 | Bebida
                2 | Fruta
                3 | Cigarro
                4 | Salgado
                5 | Doce
                6 | Laticínio
(6 rows)



### Clientes

In [ ]:
%%bash
posql -c "INSERT INTO cliente VALUES (1, 'Cesar', '111.111.111-01', 'cesinha@gmail.com');"
posql -c "INSERT INTO cliente VALUES (2, 'Igor', '222.222.222-02', 'iguinho@gmail.com');"
posql -c "INSERT INTO cliente VALUES (3, 'Diogo', '333.333.333-03', 'dioguinho@gmail.com');"
posql -c "INSERT INTO cliente VALUES (4, 'Lazaro', '444.444.444-04', 'lazinho@gmail.com');"
posql -c "INSERT INTO cliente VALUES (5, 'Maria', '555.555.555-05', 'maria@gmail.com');"
posql -c "INSERT INTO cliente VALUES (6, 'João', '666.666.666-06', 'joao@gmail.com');"
posql -c "INSERT INTO cliente VALUES (7, 'Ana', '777.777.777-07', 'ana@gmail.com');"
posql -c "INSERT INTO cliente VALUES (8, 'Pedro', '888.888.888-08', 'pedro@gmail.com');"


ERROR:  duplicate key value violates unique constraint "cliente_pkey"
DETAIL:  Key (codigo_cliente)=(1) already exists.
ERROR:  duplicate key value violates unique constraint "cliente_pkey"
DETAIL:  Key (codigo_cliente)=(2) already exists.
ERROR:  duplicate key value violates unique constraint "cliente_pkey"
DETAIL:  Key (codigo_cliente)=(3) already exists.
ERROR:  duplicate key value violates unique constraint "cliente_pkey"
DETAIL:  Key (codigo_cliente)=(4) already exists.
ERROR:  duplicate key value violates unique constraint "cliente_pkey"
DETAIL:  Key (codigo_cliente)=(5) already exists.
ERROR:  duplicate key value violates unique constraint "cliente_pkey"
DETAIL:  Key (codigo_cliente)=(6) already exists.
ERROR:  duplicate key value violates unique constraint "cliente_pkey"
DETAIL:  Key (codigo_cliente)=(7) already exists.
ERROR:  duplicate key value violates unique constraint "cliente_pkey"
DETAIL:  Key (codigo_cliente)=(8) already exists.


In [ ]:
!posql -c "SELECT * FROM cliente;"

 codigo_cliente |  nome  |    cpf_cnpj    |        email        
----------------+--------+----------------+---------------------
              1 | Cesar  | 111.111.111-01 | cesinha@gmail.com
              2 | Igor   | 222.222.222-02 | iguinho@gmail.com
              3 | Diogo  | 333.333.333-03 | dioguinho@gmail.com
              4 | Lazaro | 444.444.444-04 | lazinho@gmail.com
              5 | Maria  | 555.555.555-05 | maria@gmail.com
              6 | João   | 666.666.666-06 | joao@gmail.com
              7 | Ana    | 777.777.777-07 | ana@gmail.com
              8 | Pedro  | 888.888.888-08 | pedro@gmail.com
(8 rows)



### Fornecedores

In [ ]:
%%bash
posql -c "INSERT INTO fornecedor VALUES (1, 'Distribuidora Norte', '11.111.111/0001-01', '84999990001');"
posql -c "INSERT INTO fornecedor VALUES (2, 'Tabacaria Central', '22.222.222/0001-02', '84999990002');"
posql -c "INSERT INTO fornecedor VALUES (3, 'Doces & Cia', '33.333.333/0001-03', '84999990003');"
posql -c "INSERT INTO fornecedor VALUES (4, 'Fazenda Boa Vista', '44.444.444/0001-04', '84999990004');"


ERROR:  duplicate key value violates unique constraint "fornecedor_pkey"
DETAIL:  Key (codigo_fornecedor)=(1) already exists.
ERROR:  duplicate key value violates unique constraint "fornecedor_pkey"
DETAIL:  Key (codigo_fornecedor)=(2) already exists.
ERROR:  duplicate key value violates unique constraint "fornecedor_pkey"
DETAIL:  Key (codigo_fornecedor)=(3) already exists.
ERROR:  duplicate key value violates unique constraint "fornecedor_pkey"
DETAIL:  Key (codigo_fornecedor)=(4) already exists.


In [ ]:
!posql -c "SELECT * FROM fornecedor;"

 codigo_fornecedor |    razao_social     |        cnpj        |  telefone   
-------------------+---------------------+--------------------+-------------
                 1 | Distribuidora Norte | 11.111.111/0001-01 | 84999990001
                 2 | Tabacaria Central   | 22.222.222/0001-02 | 84999990002
                 3 | Doces & Cia         | 33.333.333/0001-03 | 84999990003
                 4 | Fazenda Boa Vista   | 44.444.444/0001-04 | 84999990004
(4 rows)



### Produtos

In [ ]:
%%bash
posql -c "INSERT INTO produto VALUES (1, 'Cafézinho', 2.00, 100, 1);"
posql -c "INSERT INTO produto VALUES (2, 'Maçã', 7.00, 10, 2);"
posql -c "INSERT INTO produto VALUES (3, 'Bad moro', 50.00, 100, 3);"
posql -c "INSERT INTO produto VALUES (4, 'Coxinha', 2.50, 5, 4);"
posql -c "INSERT INTO produto VALUES (5, 'Guaraná', 4.00, 50, 1);"
posql -c "INSERT INTO produto VALUES (6, 'Bolo de rolo', 5.00, 20, 5);"
posql -c "INSERT INTO produto VALUES (7, 'Queijo coalho', 3.50, 30, 6);"
posql -c "INSERT INTO produto VALUES (8, 'Pastel', 6.00, 15, 4);"
posql -c "INSERT INTO produto VALUES (9, 'Água', 2.00, 200, 1);"
posql -c "INSERT INTO produto VALUES (10, 'Paçoca', 1.50, 40, 5);"


ERROR:  duplicate key value violates unique constraint "produto_pkey"
DETAIL:  Key (codigo_produto)=(1) already exists.
ERROR:  duplicate key value violates unique constraint "produto_pkey"
DETAIL:  Key (codigo_produto)=(2) already exists.
ERROR:  duplicate key value violates unique constraint "produto_pkey"
DETAIL:  Key (codigo_produto)=(3) already exists.
ERROR:  duplicate key value violates unique constraint "produto_pkey"
DETAIL:  Key (codigo_produto)=(4) already exists.
ERROR:  duplicate key value violates unique constraint "produto_pkey"
DETAIL:  Key (codigo_produto)=(5) already exists.
ERROR:  duplicate key value violates unique constraint "produto_pkey"
DETAIL:  Key (codigo_produto)=(6) already exists.
ERROR:  duplicate key value violates unique constraint "produto_pkey"
DETAIL:  Key (codigo_produto)=(7) already exists.
ERROR:  duplicate key value violates unique constraint "produto_pkey"
DETAIL:  Key (codigo_produto)=(8) already exists.
ERROR:  duplicate key value violates uni

In [ ]:
!posql -c "SELECT * FROM produto;"

 codigo_produto |     nome      | preco_venda_atual | quantidade_estoque | codigo_categoria 
----------------+---------------+-------------------+--------------------+------------------
              1 | Cafézinho     |              2.00 |                100 |                1
              2 | Maçã          |              7.00 |                 10 |                2
              3 | Bad moro      |             50.00 |                100 |                3
              4 | Coxinha       |              2.50 |                  5 |                4
              5 | Guaraná       |              4.00 |                 50 |                1
              6 | Bolo de rolo  |              5.00 |                 20 |                5
              7 | Queijo coalho |              3.50 |                 30 |                6
              8 | Pastel        |              6.00 |                 15 |                4
              9 | Água          |              2.00 |                200 |    

### Vendas

In [ ]:
%%bash
posql -c "INSERT INTO venda VALUES (1, '2026-10-10 08:30:00', 6.50, 'PAGA', 1);"
posql -c "INSERT INTO venda VALUES (2, '2026-10-10 09:15:00', 52.00, 'PAGA', 3);"
posql -c "INSERT INTO venda VALUES (3, '2026-10-11 10:00:00', 25.00, 'PAGA', 2);"
posql -c "INSERT INTO venda VALUES (4, '2026-10-11 14:20:00', 7.00, 'PAGA', 4);"
posql -c "INSERT INTO venda VALUES (5, '2026-10-12 07:45:00', 9.00, 'PAGA', 5);"
posql -c "INSERT INTO venda VALUES (6, '2026-10-12 16:00:00', 14.00, 'PENDENTE', 1);"
posql -c "INSERT INTO venda VALUES (7, '2026-10-13 11:30:00', 9.50, 'PAGA', 6);"
posql -c "INSERT INTO venda VALUES (8, '2026-10-14 09:00:00', 6.50, 'CANCELADA', 7);"


ERROR:  duplicate key value violates unique constraint "venda_pkey"
DETAIL:  Key (codigo_venda)=(1) already exists.
ERROR:  duplicate key value violates unique constraint "venda_pkey"
DETAIL:  Key (codigo_venda)=(2) already exists.
ERROR:  duplicate key value violates unique constraint "venda_pkey"
DETAIL:  Key (codigo_venda)=(3) already exists.
ERROR:  duplicate key value violates unique constraint "venda_pkey"
DETAIL:  Key (codigo_venda)=(4) already exists.
ERROR:  duplicate key value violates unique constraint "venda_pkey"
DETAIL:  Key (codigo_venda)=(5) already exists.
ERROR:  duplicate key value violates unique constraint "venda_pkey"
DETAIL:  Key (codigo_venda)=(6) already exists.
ERROR:  duplicate key value violates unique constraint "venda_pkey"
DETAIL:  Key (codigo_venda)=(7) already exists.
ERROR:  duplicate key value violates unique constraint "venda_pkey"
DETAIL:  Key (codigo_venda)=(8) already exists.


In [ ]:
!posql -c "SELECT * FROM venda;"

 codigo_venda |      data_hora      | valor_total | status_venda | codigo_cliente 
--------------+---------------------+-------------+--------------+----------------
            1 | 2026-10-10 08:30:00 |        6.50 | PAGA         |              1
            2 | 2026-10-10 09:15:00 |       52.00 | PAGA         |              3
            3 | 2026-10-11 10:00:00 |       25.00 | PAGA         |              2
            4 | 2026-10-11 14:20:00 |        7.00 | PAGA         |              4
            5 | 2026-10-12 07:45:00 |        9.00 | PAGA         |              5
            6 | 2026-10-12 16:00:00 |       14.00 | PENDENTE     |              1
            7 | 2026-10-13 11:30:00 |        9.50 | PAGA         |              6
            8 | 2026-10-14 09:00:00 |        6.50 | CANCELADA    |              7
(8 rows)



### Itens das vendas

In [ ]:
%%bash
posql -c "INSERT INTO item_venda VALUES (1, 1, 2, 2.00);"   # 2 cafézinhos
posql -c "INSERT INTO item_venda VALUES (1, 4, 1, 2.50);"   # 1 coxinha
posql -c "INSERT INTO item_venda VALUES (2, 1, 1, 2.00);"   # 1 cafézinho
posql -c "INSERT INTO item_venda VALUES (2, 3, 1, 50.00);"  # 1 bad moro
posql -c "INSERT INTO item_venda VALUES (3, 2, 3, 7.00);"   # 3 maçãs
posql -c "INSERT INTO item_venda VALUES (3, 5, 1, 4.00);"   # 1 guaraná
posql -c "INSERT INTO item_venda VALUES (4, 4, 2, 2.50);"   # 2 coxinhas
posql -c "INSERT INTO item_venda VALUES (4, 9, 1, 2.00);"   # 1 água
posql -c "INSERT INTO item_venda VALUES (5, 7, 2, 3.50);"   # 2 queijo coalho
posql -c "INSERT INTO item_venda VALUES (5, 1, 1, 2.00);"   # 1 cafézinho
posql -c "INSERT INTO item_venda VALUES (6, 8, 1, 6.00);"   # 1 pastel
posql -c "INSERT INTO item_venda VALUES (6, 5, 2, 4.00);"   # 2 guaranás
posql -c "INSERT INTO item_venda VALUES (7, 6, 1, 5.00);"   # 1 bolo de rolo
posql -c "INSERT INTO item_venda VALUES (7, 10, 3, 1.50);"  # 3 paçocas
posql -c "INSERT INTO item_venda VALUES (8, 9, 2, 2.00);"   # 2 águas
posql -c "INSERT INTO item_venda VALUES (8, 4, 1, 2.50);"   # 1 coxinha


ERROR:  duplicate key value violates unique constraint "item_venda_pkey"
DETAIL:  Key (codigo_venda, codigo_produto)=(1, 1) already exists.
ERROR:  duplicate key value violates unique constraint "item_venda_pkey"
DETAIL:  Key (codigo_venda, codigo_produto)=(1, 4) already exists.
ERROR:  duplicate key value violates unique constraint "item_venda_pkey"
DETAIL:  Key (codigo_venda, codigo_produto)=(2, 1) already exists.
ERROR:  duplicate key value violates unique constraint "item_venda_pkey"
DETAIL:  Key (codigo_venda, codigo_produto)=(2, 3) already exists.
ERROR:  duplicate key value violates unique constraint "item_venda_pkey"
DETAIL:  Key (codigo_venda, codigo_produto)=(3, 2) already exists.
ERROR:  duplicate key value violates unique constraint "item_venda_pkey"
DETAIL:  Key (codigo_venda, codigo_produto)=(3, 5) already exists.
ERROR:  duplicate key value violates unique constraint "item_venda_pkey"
DETAIL:  Key (codigo_venda, codigo_produto)=(4, 4) already exists.
ERROR:  duplicate ke

In [ ]:
!posql -c "SELECT * FROM item_venda;"

 codigo_venda | codigo_produto | quantidade | preco_unitario_gravado 
--------------+----------------+------------+------------------------
            1 |              1 |          2 |                   2.00
            1 |              4 |          1 |                   2.50
            2 |              1 |          1 |                   2.00
            2 |              3 |          1 |                  50.00
            3 |              2 |          3 |                   7.00
            3 |              5 |          1 |                   4.00
            4 |              4 |          2 |                   2.50
            4 |              9 |          1 |                   2.00
            5 |              7 |          2 |                   3.50
            5 |              1 |          1 |                   2.00
            6 |              8 |          1 |                   6.00
            6 |              5 |          2 |                   4.00
            7 |              6 |

### Compras (abastecimento de estoque)

In [ ]:
%%bash
posql -c "INSERT INTO compra VALUES (1, 1, 1, 200, 0.80, '2026-10-01');"
posql -c "INSERT INTO compra VALUES (2, 1, 5, 100, 2.00, '2026-10-01');"
posql -c "INSERT INTO compra VALUES (3, 1, 9, 300, 0.50, '2026-10-01');"
posql -c "INSERT INTO compra VALUES (4, 2, 3, 150, 30.00, '2026-10-02');"
posql -c "INSERT INTO compra VALUES (5, 4, 2, 50, 3.00, '2026-10-03');"
posql -c "INSERT INTO compra VALUES (6, 4, 7, 60, 1.50, '2026-10-03');"
posql -c "INSERT INTO compra VALUES (7, 3, 6, 40, 2.50, '2026-10-04');"
posql -c "INSERT INTO compra VALUES (8, 3, 10, 80, 0.60, '2026-10-04');"
posql -c "INSERT INTO compra VALUES (9, 1, 4, 30, 1.00, '2026-10-05');"
posql -c "INSERT INTO compra VALUES (10, 1, 8, 40, 2.50, '2026-10-05');"


ERROR:  duplicate key value violates unique constraint "compra_pkey"
DETAIL:  Key (codigo_compra)=(1) already exists.
ERROR:  duplicate key value violates unique constraint "compra_pkey"
DETAIL:  Key (codigo_compra)=(2) already exists.
ERROR:  duplicate key value violates unique constraint "compra_pkey"
DETAIL:  Key (codigo_compra)=(3) already exists.
ERROR:  duplicate key value violates unique constraint "compra_pkey"
DETAIL:  Key (codigo_compra)=(4) already exists.
ERROR:  duplicate key value violates unique constraint "compra_pkey"
DETAIL:  Key (codigo_compra)=(5) already exists.
ERROR:  duplicate key value violates unique constraint "compra_pkey"
DETAIL:  Key (codigo_compra)=(6) already exists.
ERROR:  duplicate key value violates unique constraint "compra_pkey"
DETAIL:  Key (codigo_compra)=(7) already exists.
ERROR:  duplicate key value violates unique constraint "compra_pkey"
DETAIL:  Key (codigo_compra)=(8) already exists.
ERROR:  duplicate key value violates unique constraint "

In [ ]:
!posql -c "SELECT * FROM compra;"

 codigo_compra | codigo_fornecedor | codigo_produto | quantidade_comprada | preco_custo_unitario | data_compra 
---------------+-------------------+----------------+---------------------+----------------------+-------------
             1 |                 1 |              1 |                 200 |                 0.80 | 2026-10-01
             2 |                 1 |              5 |                 100 |                 2.00 | 2026-10-01
             3 |                 1 |              9 |                 300 |                 0.50 | 2026-10-01
             4 |                 2 |              3 |                 150 |                30.00 | 2026-10-02
             5 |                 4 |              2 |                  50 |                 3.00 | 2026-10-03
             6 |                 4 |              7 |                  60 |                 1.50 | 2026-10-03
             7 |                 3 |              6 |                  40 |                 2.50 | 2026-10-04
      

### Pagamentos

In [ ]:
%%bash
posql -c "INSERT INTO pagamento_venda VALUES (1, 1, 'PIX', 6.50);"
posql -c "INSERT INTO pagamento_venda VALUES (2, 2, 'DINHEIRO', 52.00);"
posql -c "INSERT INTO pagamento_venda VALUES (3, 3, 'PIX', 25.00);"
posql -c "INSERT INTO pagamento_venda VALUES (4, 4, 'DINHEIRO', 7.00);"
posql -c "INSERT INTO pagamento_venda VALUES (5, 5, 'PIX', 9.00);"
posql -c "INSERT INTO pagamento_venda VALUES (6, 7, 'DINHEIRO', 5.00);"
posql -c "INSERT INTO pagamento_venda VALUES (7, 7, 'PIX', 4.50);"


ERROR:  duplicate key value violates unique constraint "pagamento_venda_pkey"
DETAIL:  Key (codigo_pagamento)=(1) already exists.
ERROR:  duplicate key value violates unique constraint "pagamento_venda_pkey"
DETAIL:  Key (codigo_pagamento)=(2) already exists.
ERROR:  duplicate key value violates unique constraint "pagamento_venda_pkey"
DETAIL:  Key (codigo_pagamento)=(3) already exists.
ERROR:  duplicate key value violates unique constraint "pagamento_venda_pkey"
DETAIL:  Key (codigo_pagamento)=(4) already exists.
ERROR:  duplicate key value violates unique constraint "pagamento_venda_pkey"
DETAIL:  Key (codigo_pagamento)=(5) already exists.
ERROR:  duplicate key value violates unique constraint "pagamento_venda_pkey"
DETAIL:  Key (codigo_pagamento)=(6) already exists.
ERROR:  duplicate key value violates unique constraint "pagamento_venda_pkey"
DETAIL:  Key (codigo_pagamento)=(7) already exists.


Vendas 6 (PENDENTE) e 8 (CANCELADA) não possuem pagamento. Venda 7 possui dois pagamentos (DINHEIRO + PIX) para demonstrar a relação 1:N.

In [ ]:
!posql -c "SELECT * FROM pagamento_venda;"

 codigo_pagamento | codigo_venda | forma_pagamento | valor_pago 
------------------+--------------+-----------------+------------
                1 |            1 | PIX             |       6.50
                2 |            2 | DINHEIRO        |      52.00
                3 |            3 | PIX             |      25.00
                4 |            4 | DINHEIRO        |       7.00
                5 |            5 | PIX             |       9.00
                6 |            7 | DINHEIRO        |       5.00
                7 |            7 | PIX             |       4.50
(7 rows)



## Consultas de exemplo

### Listar todos os produtos com sua categoria

In [ ]:
%%bash
posql -c "SELECT p.nome AS produto, p.preco_venda_atual AS preco, c.nome AS categoria
FROM produto p
INNER JOIN categoria c ON p.codigo_categoria = c.codigo_categoria
ORDER BY c.nome, p.nome;"


    produto    | preco | categoria 
---------------+-------+-----------
 Água          |  2.00 | Bebida
 Cafézinho     |  2.00 | Bebida
 Guaraná       |  4.00 | Bebida
 Bad moro      | 50.00 | Cigarro
 Bolo de rolo  |  5.00 | Doce
 Paçoca        |  1.50 | Doce
 Maçã          |  7.00 | Fruta
 Queijo coalho |  3.50 | Laticínio
 Coxinha       |  2.50 | Salgado
 Pastel        |  6.00 | Salgado
(10 rows)



### Detalhes de cada venda (cliente + itens)

In [ ]:
%%bash
posql -c "SELECT v.codigo_venda, cl.nome AS cliente, v.data_hora,
       p.nome AS produto, iv.quantidade, iv.preco_unitario_gravado,
       (iv.quantidade * iv.preco_unitario_gravado) AS subtotal
FROM venda v
INNER JOIN cliente cl ON v.codigo_cliente = cl.codigo_cliente
INNER JOIN item_venda iv ON v.codigo_venda = iv.codigo_venda
INNER JOIN produto p ON iv.codigo_produto = p.codigo_produto
ORDER BY v.codigo_venda;"


 codigo_venda | cliente |      data_hora      |    produto    | quantidade | preco_unitario_gravado | subtotal 
--------------+---------+---------------------+---------------+------------+------------------------+----------
            1 | Cesar   | 2026-10-10 08:30:00 | Cafézinho     |          2 |                   2.00 |     4.00
            1 | Cesar   | 2026-10-10 08:30:00 | Coxinha       |          1 |                   2.50 |     2.50
            2 | Diogo   | 2026-10-10 09:15:00 | Cafézinho     |          1 |                   2.00 |     2.00
            2 | Diogo   | 2026-10-10 09:15:00 | Bad moro      |          1 |                  50.00 |    50.00
            3 | Igor    | 2026-10-11 10:00:00 | Maçã          |          3 |                   7.00 |    21.00
            3 | Igor    | 2026-10-11 10:00:00 | Guaraná       |          1 |                   4.00 |     4.00
            4 | Lazaro  | 2026-10-11 14:20:00 | Coxinha       |          2 |                   2.50 |     5.00

### Total de cada venda

In [ ]:
%%bash
posql -c "SELECT v.codigo_venda, cl.nome AS cliente,
       SUM(iv.quantidade * iv.preco_unitario_gravado) AS total
FROM venda v
INNER JOIN cliente cl ON v.codigo_cliente = cl.codigo_cliente
INNER JOIN item_venda iv ON v.codigo_venda = iv.codigo_venda
GROUP BY v.codigo_venda, cl.nome
ORDER BY v.codigo_venda;"


 codigo_venda | cliente | total 
--------------+---------+-------
            1 | Cesar   |  6.50
            2 | Diogo   | 52.00
            3 | Igor    | 25.00
            4 | Lazaro  |  7.00
            5 | Maria   |  9.00
            6 | Cesar   | 14.00
            7 | João    |  9.50
            8 | Ana     |  6.50
(8 rows)



### Produto mais vendido (por quantidade)

In [ ]:
%%bash
posql -c "SELECT p.nome, SUM(iv.quantidade) AS total_vendido
FROM item_venda iv
INNER JOIN produto p ON iv.codigo_produto = p.codigo_produto
GROUP BY p.nome
ORDER BY total_vendido DESC;"


     nome      | total_vendido 
---------------+---------------
 Cafézinho     |             4
 Coxinha       |             4
 Maçã          |             3
 Paçoca        |             3
 Guaraná       |             3
 Água          |             3
 Queijo coalho |             2
 Bad moro      |             1
 Bolo de rolo  |             1
 Pastel        |             1
(10 rows)



### Faturamento por categoria

In [ ]:
%%bash
posql -c "SELECT c.nome AS categoria,
       SUM(iv.quantidade * iv.preco_unitario_gravado) AS faturamento
FROM item_venda iv
INNER JOIN produto p ON iv.codigo_produto = p.codigo_produto
INNER JOIN categoria c ON p.codigo_categoria = c.codigo_categoria
GROUP BY c.nome
ORDER BY faturamento DESC;"


 categoria | faturamento 
-----------+-------------
 Cigarro   |       50.00
 Bebida    |       26.00
 Fruta     |       21.00
 Salgado   |       16.00
 Doce      |        9.50
 Laticínio |        7.00
(6 rows)



### Lucro por produto (venda vs custo)

In [ ]:
%%bash
posql -c "WITH custo_medio AS (
    SELECT codigo_produto,
           SUM(quantidade_comprada * preco_custo_unitario) / SUM(quantidade_comprada) AS custo_unitario_medio
    FROM compra
    GROUP BY codigo_produto
),
receita AS (
    SELECT iv.codigo_produto,
           SUM(iv.quantidade) AS qtd_vendida,
           SUM(iv.quantidade * iv.preco_unitario_gravado) AS receita_total
    FROM item_venda iv
    GROUP BY iv.codigo_produto
)
SELECT p.nome,
       r.qtd_vendida,
       ROUND(r.receita_total, 2) AS receita,
       ROUND(r.qtd_vendida * cm.custo_unitario_medio, 2) AS custo_total,
       ROUND(r.receita_total - (r.qtd_vendida * cm.custo_unitario_medio), 2) AS lucro
FROM receita r
INNER JOIN produto p ON r.codigo_produto = p.codigo_produto
INNER JOIN custo_medio cm ON r.codigo_produto = cm.codigo_produto
ORDER BY lucro DESC;"


     nome      | qtd_vendida | receita | custo_total | lucro 
---------------+-------------+---------+-------------+-------
 Bad moro      |           1 |   50.00 |       30.00 | 20.00
 Maçã          |           3 |   21.00 |        9.00 | 12.00
 Coxinha       |           4 |   10.00 |        4.00 |  6.00
 Guaraná       |           3 |   12.00 |        6.00 |  6.00
 Cafézinho     |           4 |    8.00 |        3.20 |  4.80
 Água          |           3 |    6.00 |        1.50 |  4.50
 Queijo coalho |           2 |    7.00 |        3.00 |  4.00
 Pastel        |           1 |    6.00 |        2.50 |  3.50
 Paçoca        |           3 |    4.50 |        1.80 |  2.70
 Bolo de rolo  |           1 |    5.00 |        2.50 |  2.50
(10 rows)



### Pagamentos por forma de pagamento

In [ ]:
%%bash
posql -c "SELECT forma_pagamento,
       COUNT(*) AS qtd_pagamentos,
       SUM(valor_pago) AS total_recebido
FROM pagamento_venda
GROUP BY forma_pagamento
ORDER BY total_recebido DESC;"


 forma_pagamento | qtd_pagamentos | total_recebido 
-----------------+----------------+----------------
 DINHEIRO        |              3 |          64.00
 PIX             |              4 |          45.00
(2 rows)



### Fornecedores e seus produtos fornecidos

In [ ]:
%%bash
posql -c "SELECT f.razao_social AS fornecedor,
       p.nome AS produto,
       SUM(co.quantidade_comprada) AS total_fornecido,
       ROUND(AVG(co.preco_custo_unitario), 2) AS preco_custo_medio
FROM compra co
INNER JOIN fornecedor f ON co.codigo_fornecedor = f.codigo_fornecedor
INNER JOIN produto p ON co.codigo_produto = p.codigo_produto
GROUP BY f.razao_social, p.nome
ORDER BY f.razao_social, p.nome;"


     fornecedor      |    produto    | total_fornecido | preco_custo_medio 
---------------------+---------------+-----------------+-------------------
 Distribuidora Norte | Água          |             300 |              0.50
 Distribuidora Norte | Cafézinho     |             200 |              0.80
 Distribuidora Norte | Coxinha       |              30 |              1.00
 Distribuidora Norte | Guaraná       |             100 |              2.00
 Distribuidora Norte | Pastel        |              40 |              2.50
 Doces & Cia         | Bolo de rolo  |              40 |              2.50
 Doces & Cia         | Paçoca        |              80 |              0.60
 Fazenda Boa Vista   | Maçã          |              50 |              3.00
 Fazenda Boa Vista   | Queijo coalho |              60 |              1.50
 Tabacaria Central   | Bad moro      |             150 |             30.00
(10 rows)



In [ ]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"

ALTER ROLE


## Aplicação — Simulação de uso real do BD

Usando `psycopg2` para conectar ao PostgreSQL e executar operações como uma aplicação faria.

In [ ]:
import psycopg2
from datetime import datetime

def conectar():
    return psycopg2.connect(
        dbname="lojadoseuze",
        user="postgres",
        password="postgres",
        host="localhost"
    )

conn = conectar()
conn.autocommit = False
print("Conectado ao banco lojadoseuze.")

Conectado ao banco lojadoseuze.


### Funções da aplicação

Cada função simula uma operação real do sistema da loja.

In [ ]:
def consultar(sql, params=None):
    """Executa um SELECT e retorna os resultados."""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        colunas = [desc[0] for desc in cur.description]
        linhas = cur.fetchall()
    return colunas, linhas


def mostrar(colunas, linhas):
    """Exibe resultados formatados."""
    larguras = [len(c) for c in colunas]
    for linha in linhas:
        for i, val in enumerate(linha):
            larguras[i] = max(larguras[i], len(str(val)))

    header = " | ".join(c.ljust(larguras[i]) for i, c in enumerate(colunas))
    separador = "-+-".join("-" * l for l in larguras)
    print(header)
    print(separador)
    for linha in linhas:
        print(" | ".join(str(v).ljust(larguras[i]) for i, v in enumerate(linha)))
    print(f"({len(linhas)} linhas)")

In [ ]:
def ver_estoque(codigo_produto=None):
    """Mostra o estoque atual. Se passar código, filtra um produto."""
    sql = """SELECT p.codigo_produto, p.nome, p.quantidade_estoque,
                    p.preco_venda_atual, c.nome AS categoria
             FROM produto p
             INNER JOIN categoria c ON p.codigo_categoria = c.codigo_categoria"""
    params = None
    if codigo_produto:
        sql += " WHERE p.codigo_produto = %s"
        params = (codigo_produto,)
    sql += " ORDER BY p.codigo_produto"
    mostrar(*consultar(sql, params))

In [ ]:
def registrar_compra(codigo_fornecedor, codigo_produto, quantidade, preco_custo):
    """Registra uma compra de fornecedor e atualiza o estoque."""
    try:
        with conn.cursor() as cur:
            # Gera próximo código
            cur.execute("SELECT COALESCE(MAX(codigo_compra), 0) + 1 FROM compra")
            codigo = cur.fetchone()[0]

            # Insere a compra
            cur.execute(
                """INSERT INTO compra
                   (codigo_compra, codigo_fornecedor, codigo_produto,
                    quantidade_comprada, preco_custo_unitario, data_compra)
                   VALUES (%s, %s, %s, %s, %s, %s)""",
                (codigo, codigo_fornecedor, codigo_produto,
                 quantidade, preco_custo, datetime.now().date())
            )

            # Atualiza estoque
            cur.execute(
                """UPDATE produto
                   SET quantidade_estoque = quantidade_estoque + %s
                   WHERE codigo_produto = %s""",
                (quantidade, codigo_produto)
            )

        conn.commit()
        print(f"Compra #{codigo} registrada: +{quantidade} unidades do produto {codigo_produto}")
        print(f"Custo unitário: R$ {preco_custo:.2f} | Total: R$ {quantidade * preco_custo:.2f}")
    except Exception as e:
        conn.rollback()
        print(f"ERRO na compra: {e}")

In [ ]:
def registrar_venda(codigo_cliente, itens, forma_pagamento):
    """
    Registra uma venda completa: venda + itens + pagamento + baixa no estoque.

    itens = lista de tuplas (codigo_produto, quantidade)
    forma_pagamento = 'PIX', 'DINHEIRO', etc.
    """
    try:
        with conn.cursor() as cur:
            # Verifica estoque antes de tudo
            for cod_prod, qtd in itens:
                cur.execute(
                    "SELECT nome, quantidade_estoque, preco_venda_atual FROM produto WHERE codigo_produto = %s",
                    (cod_prod,)
                )
                produto = cur.fetchone()
                if not produto:
                    raise ValueError(f"Produto {cod_prod} não encontrado.")
                nome, estoque, _ = produto
                if estoque < qtd:
                    raise ValueError(f"Estoque insuficiente para '{nome}': pedido {qtd}, disponível {estoque}.")

            # Gera código da venda
            cur.execute("SELECT COALESCE(MAX(codigo_venda), 0) + 1 FROM venda")
            cod_venda = cur.fetchone()[0]

            # Calcula valor total e insere itens
            valor_total = 0
            detalhes = []
            for cod_prod, qtd in itens:
                cur.execute("SELECT nome, preco_venda_atual FROM produto WHERE codigo_produto = %s", (cod_prod,))
                nome, preco = cur.fetchone()

                cur.execute(
                    "INSERT INTO item_venda VALUES (%s, %s, %s, %s)",
                    (cod_venda, cod_prod, qtd, preco)
                )

                subtotal = qtd * preco
                valor_total += subtotal
                detalhes.append(f"  {qtd}x {nome} @ R$ {preco:.2f} = R$ {subtotal:.2f}")

                # Baixa no estoque
                cur.execute(
                    "UPDATE produto SET quantidade_estoque = quantidade_estoque - %s WHERE codigo_produto = %s",
                    (qtd, cod_prod)
                )

            # Insere a venda
            cur.execute(
                "INSERT INTO venda VALUES (%s, %s, %s, %s, %s)",
                (cod_venda, datetime.now(), valor_total, 'PAGA', codigo_cliente)
            )

            # Insere o pagamento
            cur.execute("SELECT COALESCE(MAX(codigo_pagamento), 0) + 1 FROM pagamento_venda")
            cod_pag = cur.fetchone()[0]
            cur.execute(
                "INSERT INTO pagamento_venda VALUES (%s, %s, %s, %s)",
                (cod_pag, cod_venda, forma_pagamento, valor_total)
            )

        conn.commit()
        print(f"=== VENDA #{cod_venda} ===")
        print(f"Cliente: {codigo_cliente}")
        for d in detalhes:
            print(d)
        print(f"  TOTAL: R$ {valor_total:.2f}")
        print(f"  Pagamento: {forma_pagamento}")
        print("  Status: PAGA")
    except Exception as e:
        conn.rollback()
        print(f"ERRO na venda: {e}")

In [ ]:
def cancelar_venda(codigo_venda):
    """Cancela uma venda: muda status e devolve itens ao estoque."""
    try:
        with conn.cursor() as cur:
            cur.execute("SELECT status_venda FROM venda WHERE codigo_venda = %s", (codigo_venda,))
            resultado = cur.fetchone()
            if not resultado:
                raise ValueError(f"Venda {codigo_venda} não encontrada.")
            if resultado[0] == 'CANCELADA':
                raise ValueError(f"Venda {codigo_venda} já está cancelada.")

            # Devolve ao estoque
            cur.execute(
                "SELECT codigo_produto, quantidade FROM item_venda WHERE codigo_venda = %s",
                (codigo_venda,)
            )
            for cod_prod, qtd in cur.fetchall():
                cur.execute(
                    "UPDATE produto SET quantidade_estoque = quantidade_estoque + %s WHERE codigo_produto = %s",
                    (qtd, cod_prod)
                )

            # Remove pagamentos e muda status
            cur.execute("DELETE FROM pagamento_venda WHERE codigo_venda = %s", (codigo_venda,))
            cur.execute("UPDATE venda SET status_venda = 'CANCELADA' WHERE codigo_venda = %s", (codigo_venda,))

        conn.commit()
        print(f"Venda #{codigo_venda} cancelada. Estoque devolvido.")
    except Exception as e:
        conn.rollback()
        print(f"ERRO ao cancelar: {e}")

In [ ]:
def relatorio_lucro():
    """Relatório de lucro por produto (receita - custo)."""
    sql = """
    WITH custo_medio AS (
        SELECT codigo_produto,
               SUM(quantidade_comprada * preco_custo_unitario) / SUM(quantidade_comprada) AS custo_unit
        FROM compra
        GROUP BY codigo_produto
    ),
    receita AS (
        SELECT iv.codigo_produto,
               SUM(iv.quantidade) AS qtd_vendida,
               SUM(iv.quantidade * iv.preco_unitario_gravado) AS receita_total
        FROM item_venda iv
        INNER JOIN venda v ON iv.codigo_venda = v.codigo_venda
        WHERE v.status_venda != 'CANCELADA'
        GROUP BY iv.codigo_produto
    )
    SELECT p.nome,
           r.qtd_vendida,
           ROUND(r.receita_total, 2) AS receita,
           ROUND(r.qtd_vendida * cm.custo_unit, 2) AS custo,
           ROUND(r.receita_total - (r.qtd_vendida * cm.custo_unit), 2) AS lucro
    FROM receita r
    INNER JOIN produto p ON r.codigo_produto = p.codigo_produto
    INNER JOIN custo_medio cm ON r.codigo_produto = cm.codigo_produto
    ORDER BY lucro DESC
    """
    mostrar(*consultar(sql))

### Demonstração

#### 1. Ver estoque atual

In [ ]:
ver_estoque()

codigo_produto | nome          | quantidade_estoque | preco_venda_atual | categoria
---------------+---------------+--------------------+-------------------+----------
1              | Cafézinho     | 100                | 2.00              | Bebida   
2              | Maçã          | 10                 | 7.00              | Fruta    
3              | Bad moro      | 100                | 50.00             | Cigarro  
4              | Coxinha       | 5                  | 2.50              | Salgado  
5              | Guaraná       | 50                 | 4.00              | Bebida   
6              | Bolo de rolo  | 20                 | 5.00              | Doce     
7              | Queijo coalho | 30                 | 3.50              | Laticínio
8              | Pastel        | 15                 | 6.00              | Salgado  
9              | Água          | 200                | 2.00              | Bebida   
10             | Paçoca        | 40                 | 1.50              | Do

#### 2. Registrar uma compra de fornecedor (abastecer estoque)

In [ ]:
# Comprando 50 coxinhas da Distribuidora Norte a R$ 1.00 cada
registrar_compra(
    codigo_fornecedor=1,
    codigo_produto=4,
    quantidade=50,
    preco_custo=1.00
)

Compra #11 registrada: +50 unidades do produto 4
Custo unitário: R$ 1.00 | Total: R$ 50.00


In [ ]:
# Conferir estoque do produto 4 (Coxinha)
ver_estoque(4)

codigo_produto | nome    | quantidade_estoque | preco_venda_atual | categoria
---------------+---------+--------------------+-------------------+----------
4              | Coxinha | 55                 | 2.50              | Salgado  
(1 linhas)


#### 3. Registrar uma venda

In [ ]:
# Cliente 3 (Diogo) compra 2 cafézinhos + 3 coxinhas, paga no PIX
registrar_venda(
    codigo_cliente=3,
    itens=[(1, 2), (4, 3)],
    forma_pagamento='PIX'
)

ERRO na venda: insert or update on table "item_venda" violates foreign key constraint "fk_item_venda_venda"
DETAIL:  Key (codigo_venda)=(9) is not present in table "venda".



In [ ]:
# Estoque atualizado
ver_estoque()

codigo_produto | nome          | quantidade_estoque | preco_venda_atual | categoria
---------------+---------------+--------------------+-------------------+----------
1              | Cafézinho     | 100                | 2.00              | Bebida   
2              | Maçã          | 10                 | 7.00              | Fruta    
3              | Bad moro      | 100                | 50.00             | Cigarro  
4              | Coxinha       | 55                 | 2.50              | Salgado  
5              | Guaraná       | 50                 | 4.00              | Bebida   
6              | Bolo de rolo  | 20                 | 5.00              | Doce     
7              | Queijo coalho | 30                 | 3.50              | Laticínio
8              | Pastel        | 15                 | 6.00              | Salgado  
9              | Água          | 200                | 2.00              | Bebida   
10             | Paçoca        | 40                 | 1.50              | Do

#### 4. Tentar vender mais do que tem no estoque

In [ ]:
# Tentar vender 999 maçãs — deve dar erro
registrar_venda(
    codigo_cliente=1,
    itens=[(2, 999)],
    forma_pagamento='DINHEIRO'
)

ERRO na venda: Estoque insuficiente para 'Maçã': pedido 999, disponível 10.


#### 5. Cancelar uma venda

In [ ]:
# Cancelar a venda 6 (que estava PENDENTE)
cancelar_venda(6)

Venda #6 cancelada. Estoque devolvido.


In [ ]:
# Conferir que o estoque voltou
ver_estoque()

codigo_produto | nome          | quantidade_estoque | preco_venda_atual | categoria
---------------+---------------+--------------------+-------------------+----------
1              | Cafézinho     | 100                | 2.00              | Bebida   
2              | Maçã          | 10                 | 7.00              | Fruta    
3              | Bad moro      | 100                | 50.00             | Cigarro  
4              | Coxinha       | 55                 | 2.50              | Salgado  
5              | Guaraná       | 52                 | 4.00              | Bebida   
6              | Bolo de rolo  | 20                 | 5.00              | Doce     
7              | Queijo coalho | 30                 | 3.50              | Laticínio
8              | Pastel        | 16                 | 6.00              | Salgado  
9              | Água          | 200                | 2.00              | Bebida   
10             | Paçoca        | 40                 | 1.50              | Do

#### 6. Relatório de lucro

In [ ]:
relatorio_lucro()

nome          | qtd_vendida | receita | custo | lucro
--------------+-------------+---------+-------+------
Bad moro      | 1           | 50.00   | 30.00 | 20.00
Maçã          | 3           | 21.00   | 9.00  | 12.00
Cafézinho     | 4           | 8.00    | 3.20  | 4.80 
Coxinha       | 3           | 7.50    | 3.00  | 4.50 
Queijo coalho | 2           | 7.00    | 3.00  | 4.00 
Paçoca        | 3           | 4.50    | 1.80  | 2.70 
Bolo de rolo  | 1           | 5.00    | 2.50  | 2.50 
Guaraná       | 1           | 4.00    | 2.00  | 2.00 
Água          | 1           | 2.00    | 0.50  | 1.50 
(9 linhas)


#### 7. Encerrar conexão

In [ ]:
conn.close()
print('Conexão encerrada.')

Conexão encerrada.
